# Self-Attention Generative Adversarial Network (SAGAN)

This notebook implements the **Self-Attention GAN (SAGAN)** architecture for image generation.

**Reference:** Zhang, H., Goodfellow, I., Metaxas, D., & Odena, A. (2019). *Self-Attention Generative Adversarial Networks.* ICML 2019.

## Key Concepts
- **Self-Attention Mechanism:** Allows the model to capture long-range dependencies in the image, enabling more globally coherent generation.
- **Spectral Normalization:** Applied to both Generator and Discriminator to stabilize training.
- **Hinge Loss:** Used as the adversarial loss function for more stable GAN training.

---
## 1. Imports and Environment Setup

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from torch.nn.utils import spectral_norm

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## 2. Hyperparameters and Configuration

In [ ]:
# ── Training settings ──────────────────────────────────────────────────────────
BATCH_SIZE    = 64        # Number of images per training batch
NUM_EPOCHS    = 100       # Total number of training epochs
LEARNING_RATE = 0.0001    # Learning rate for both Generator and Discriminator
BETA1         = 0.0       # Adam optimizer beta1
BETA2         = 0.9       # Adam optimizer beta2

# ── Model architecture ─────────────────────────────────────────────────────────
Z_DIM         = 128       # Dimensionality of the latent noise vector
G_CONV_DIM    = 64        # Base number of channels in the Generator
D_CONV_DIM    = 64        # Base number of channels in the Discriminator
IMAGE_SIZE    = 64        # Spatial resolution of generated images (64x64)
NUM_CHANNELS  = 3         # Number of image channels (3 for RGB)

# ── Discriminator update frequency ────────────────────────────────────────────
N_CRITIC      = 1         # Number of Discriminator updates per Generator update

# ── Logging & checkpointing ────────────────────────────────────────────────────
SAVE_INTERVAL = 10        # Save sample images every N epochs
CHECKPOINT_DIR = 'checkpoints'
SAMPLE_DIR     = 'samples'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

print('Configuration loaded successfully.')

---
## 3. Data Loading and Preprocessing

In [ ]:
# Image transformation pipeline
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Load dataset (using CelebA as example; replace with your own dataset path)
dataset = datasets.ImageFolder(root='data/', transform=transform)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

print(f'Dataset size: {len(dataset)} images')
print(f'Number of batches per epoch: {len(dataloader)}')

In [ ]:
# Visualize a sample batch from the dataset
def show_batch(dataloader, num_images=16):
    """Display a grid of sample images from the dataset."""
    images, _ = next(iter(dataloader))
    images = images[:num_images]
    # Denormalize from [-1, 1] to [0, 1]
    images = (images + 1) / 2
    grid = make_grid(images, nrow=4, padding=2)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title('Sample images from the dataset')
    plt.axis('off')
    plt.show()

show_batch(dataloader)

---
## 4. Self-Attention Module

The self-attention layer is the core component of SAGAN. It computes attention maps over spatial positions, allowing the network to model long-range dependencies.

For each position $i$, the attention map is computed as:
$$\beta_{j,i} = \frac{\exp(s_{ij})}{\sum_{i=1}^{N} \exp(s_{ij})}, \quad s_{ij} = f(x_i)^T g(x_j)$$

The output is a weighted sum of feature projections: $o_j = \sum_i \beta_{j,i} h(x_i)$

In [ ]:
class SelfAttention(nn.Module):
    """Self-Attention module for spatial feature maps.

    Computes attention between all pairs of spatial positions to capture
    long-range dependencies.

    Args:
        in_channels (int): Number of input feature map channels.
    """

    def __init__(self, in_channels: int):
        super(SelfAttention, self).__init__()

        # Query, Key and Value projections (reduce channels for efficiency)
        self.query = spectral_norm(nn.Conv1d(in_channels, in_channels // 8, kernel_size=1))
        self.key   = spectral_norm(nn.Conv1d(in_channels, in_channels // 8, kernel_size=1))
        self.value = spectral_norm(nn.Conv1d(in_channels, in_channels,       kernel_size=1))

        # Learnable scaling parameter (initialized to 0 so attention is disabled early in training)
        self.gamma = nn.Parameter(torch.zeros(1))

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: Input feature map of shape (B, C, H, W).
        Returns:
            out:            Attention-enhanced feature map, same shape as x.
            attention_map:  Attention weights of shape (B, N, N) where N = H*W.
        """
        B, C, H, W = x.size()
        N = H * W

        # Flatten spatial dimensions: (B, C, H, W) -> (B, C, N)
        x_flat = x.view(B, C, N)

        # Compute Query, Key, Value projections
        q = self.query(x_flat)  # (B, C/8, N)
        k = self.key(x_flat)    # (B, C/8, N)
        v = self.value(x_flat)  # (B, C,   N)

        # Compute attention scores: (B, N, C/8) x (B, C/8, N) -> (B, N, N)
        attention_map = self.softmax(torch.bmm(q.permute(0, 2, 1), k))

        # Weighted aggregation of values: (B, C, N) x (B, N, N) -> (B, C, N)
        out = torch.bmm(v, attention_map.permute(0, 2, 1))

        # Reshape back to spatial feature map
        out = out.view(B, C, H, W)

        # Residual connection with learned scaling
        out = self.gamma * out + x
        return out, attention_map

---
## 5. Generator Architecture

The Generator transforms a latent noise vector **z** into a full-resolution image through a series of transposed convolution blocks. Self-Attention layers are inserted at intermediate resolutions to enable global coherence.

In [ ]:
class GeneratorBlock(nn.Module):
    """Upsampling residual block used in the Generator.

    Applies spectral normalization and batch normalization.

    Args:
        in_channels  (int): Number of input channels.
        out_channels (int): Number of output channels.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super(GeneratorBlock, self).__init__()
        self.block = nn.Sequential(
            spectral_norm(nn.ConvTranspose2d(in_channels, out_channels,
                                             kernel_size=4, stride=2, padding=1, bias=False)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Generator(nn.Module):
    """SAGAN Generator network.

    Maps a latent vector z to a synthesized image. Self-Attention is applied
    at two intermediate feature map resolutions.

    Args:
        z_dim       (int): Dimensionality of the input noise vector.
        conv_dim    (int): Base number of feature channels.
        num_channels(int): Number of output image channels.
    """

    def __init__(self, z_dim: int = Z_DIM, conv_dim: int = G_CONV_DIM,
                 num_channels: int = NUM_CHANNELS):
        super(Generator, self).__init__()

        # Project and reshape noise vector: z_dim -> (conv_dim*8) x 4 x 4
        self.fc = spectral_norm(nn.Linear(z_dim, conv_dim * 8 * 4 * 4))
        self.conv_dim = conv_dim

        # Upsampling blocks: 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
        self.block1 = GeneratorBlock(conv_dim * 8, conv_dim * 4)  # 4  -> 8
        self.block2 = GeneratorBlock(conv_dim * 4, conv_dim * 2)  # 8  -> 16
        self.attn1  = SelfAttention(conv_dim * 2)                  # attention at 16x16
        self.block3 = GeneratorBlock(conv_dim * 2, conv_dim)       # 16 -> 32
        self.attn2  = SelfAttention(conv_dim)                      # attention at 32x32
        self.block4 = GeneratorBlock(conv_dim, conv_dim // 2)      # 32 -> 64

        # Final output layer: map to image space with Tanh activation
        self.output_conv = nn.Sequential(
            spectral_norm(nn.Conv2d(conv_dim // 2, num_channels,
                                    kernel_size=3, stride=1, padding=1)),
            nn.Tanh()
        )

    def forward(self, z: torch.Tensor):
        """
        Args:
            z: Latent noise vector of shape (B, z_dim).
        Returns:
            img:   Generated image, shape (B, num_channels, 64, 64).
            attn1: Attention map from the first self-attention layer.
            attn2: Attention map from the second self-attention layer.
        """
        # Project noise and reshape to spatial feature map
        x = self.fc(z)
        x = x.view(z.size(0), self.conv_dim * 8, 4, 4)

        # Progressive upsampling
        x = self.block1(x)
        x = self.block2(x)
        x, attn1 = self.attn1(x)
        x = self.block3(x)
        x, attn2 = self.attn2(x)
        x = self.block4(x)
        img = self.output_conv(x)
        return img, attn1, attn2

---
## 6. Discriminator Architecture

The Discriminator classifies images as real or fake. It mirrors the Generator structure using strided convolutions for downsampling, with Self-Attention layers at intermediate resolutions. Spectral normalization is applied to all convolutional layers.

In [ ]:
class DiscriminatorBlock(nn.Module):
    """Downsampling block used in the Discriminator.

    Uses spectral normalization and LeakyReLU activation.

    Args:
        in_channels  (int): Number of input channels.
        out_channels (int): Number of output channels.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super(DiscriminatorBlock, self).__init__()
        self.block = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels, out_channels,
                                    kernel_size=4, stride=2, padding=1, bias=False)),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Discriminator(nn.Module):
    """SAGAN Discriminator network.

    Classifies images as real or generated. Self-Attention is applied
    at intermediate feature map resolutions.

    Args:
        conv_dim    (int): Base number of feature channels.
        num_channels(int): Number of input image channels.
    """

    def __init__(self, conv_dim: int = D_CONV_DIM, num_channels: int = NUM_CHANNELS):
        super(Discriminator, self).__init__()

        # Downsampling blocks: 64x64 -> 32x32 -> 16x16 -> 8x8 -> 4x4
        self.block1 = DiscriminatorBlock(num_channels,   conv_dim)       # 64 -> 32
        self.block2 = DiscriminatorBlock(conv_dim,       conv_dim * 2)   # 32 -> 16
        self.attn1  = SelfAttention(conv_dim * 2)                         # attention at 16x16
        self.block3 = DiscriminatorBlock(conv_dim * 2,   conv_dim * 4)   # 16 -> 8
        self.attn2  = SelfAttention(conv_dim * 4)                         # attention at 8x8
        self.block4 = DiscriminatorBlock(conv_dim * 4,   conv_dim * 8)   # 8  -> 4

        # Final classification layer
        self.output_conv = spectral_norm(
            nn.Conv2d(conv_dim * 8, 1, kernel_size=4, stride=1, padding=0)
        )

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: Input image of shape (B, num_channels, 64, 64).
        Returns:
            out:   Scalar logit per image, shape (B, 1).
            attn1: Attention map from the first self-attention layer.
            attn2: Attention map from the second self-attention layer.
        """
        x = self.block1(x)
        x = self.block2(x)
        x, attn1 = self.attn1(x)
        x = self.block3(x)
        x, attn2 = self.attn2(x)
        x = self.block4(x)
        out = self.output_conv(x)
        out = out.view(out.size(0), -1)  # Flatten to (B, 1)
        return out, attn1, attn2

---
## 7. Loss Functions and Training Utilities

SAGAN uses the **Hinge loss** variant of the adversarial objective:

$$\mathcal{L}_D = -\mathbb{E}[\min(0, -1 + D(x))] - \mathbb{E}[\min(0, -1 - D(G(z)))]$$

$$\mathcal{L}_G = -\mathbb{E}[D(G(z))]$$

In [ ]:
def discriminator_hinge_loss(real_logits: torch.Tensor,
                             fake_logits: torch.Tensor) -> torch.Tensor:
    """Hinge loss for the Discriminator.

    Args:
        real_logits: Discriminator output for real images.
        fake_logits: Discriminator output for generated images.

    Returns:
        Scalar loss value.
    """
    real_loss = F.relu(1.0 - real_logits).mean()
    fake_loss = F.relu(1.0 + fake_logits).mean()
    return real_loss + fake_loss


def generator_hinge_loss(fake_logits: torch.Tensor) -> torch.Tensor:
    """Hinge loss for the Generator.

    Args:
        fake_logits: Discriminator output for generated images.

    Returns:
        Scalar loss value.
    """
    return -fake_logits.mean()


def weights_init(m: nn.Module):
    """Initialize Conv and BatchNorm weights following the DCGAN paper."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# Instantiate models and move to device
G = Generator(z_dim=Z_DIM, conv_dim=G_CONV_DIM, num_channels=NUM_CHANNELS).to(device)
D = Discriminator(conv_dim=D_CONV_DIM, num_channels=NUM_CHANNELS).to(device)

G.apply(weights_init)
D.apply(weights_init)

# Optimizers (two-timescale update rule: different LRs for G and D are optional)
optimizer_G = optim.Adam(G.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))
optimizer_D = optim.Adam(D.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))

print('Generator architecture:')
print(G)
print(f'\nTotal Generator parameters:     {sum(p.numel() for p in G.parameters()):,}')
print(f'Total Discriminator parameters: {sum(p.numel() for p in D.parameters()):,}')

---
## 8. Training Loop

Training alternates between updating the **Discriminator** (on real and fake batches) and the **Generator**. A fixed noise vector `z_fixed` is used throughout training to visualize progress.

In [ ]:
# Fixed noise for consistent progress visualization
z_fixed = torch.randn(64, Z_DIM, device=device)

# Lists to track loss history for plotting
g_losses = []
d_losses = []


def save_samples(epoch: int, z: torch.Tensor):
    """Generate and save a grid of sample images."""
    G.eval()
    with torch.no_grad():
        fake_images, _, _ = G(z)
    G.train()
    # Denormalize from [-1, 1] to [0, 1] before saving
    fake_images = (fake_images + 1) / 2
    save_image(fake_images, os.path.join(SAMPLE_DIR, f'epoch_{epoch:04d}.png'), nrow=8)


print('Starting training...')
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0

    for batch_idx, (real_images, _) in enumerate(dataloader):
        real_images = real_images.to(device)
        current_batch_size = real_images.size(0)

        # ── Update Discriminator ───────────────────────────────────────────────
        optimizer_D.zero_grad()

        # Real images
        real_logits, _, _ = D(real_images)

        # Generated images (detach to avoid backprop through Generator)
        z = torch.randn(current_batch_size, Z_DIM, device=device)
        fake_images, _, _ = G(z)
        fake_logits, _, _ = D(fake_images.detach())

        d_loss = discriminator_hinge_loss(real_logits, fake_logits)
        d_loss.backward()
        optimizer_D.step()

        epoch_d_loss += d_loss.item()

        # ── Update Generator ───────────────────────────────────────────────────
        if (batch_idx + 1) % N_CRITIC == 0:
            optimizer_G.zero_grad()

            z = torch.randn(current_batch_size, Z_DIM, device=device)
            fake_images, _, _ = G(z)
            fake_logits, _, _ = D(fake_images)

            g_loss = generator_hinge_loss(fake_logits)
            g_loss.backward()
            optimizer_G.step()

            epoch_g_loss += g_loss.item()

    # Record average losses for this epoch
    avg_d_loss = epoch_d_loss / len(dataloader)
    avg_g_loss = epoch_g_loss / (len(dataloader) // N_CRITIC)
    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    elapsed = time.time() - start_time
    print(f'Epoch [{epoch:>4}/{NUM_EPOCHS}]  '
          f'D Loss: {avg_d_loss:.4f}  '
          f'G Loss: {avg_g_loss:.4f}  '
          f'Elapsed: {elapsed:.1f}s')

    # Save generated samples periodically
    if epoch % SAVE_INTERVAL == 0:
        save_samples(epoch, z_fixed)
        # Save model checkpoints
        torch.save(G.state_dict(), os.path.join(CHECKPOINT_DIR, f'G_epoch_{epoch:04d}.pth'))
        torch.save(D.state_dict(), os.path.join(CHECKPOINT_DIR, f'D_epoch_{epoch:04d}.pth'))
        print(f'  -> Checkpoint and samples saved for epoch {epoch}.')

print('Training complete.')

---
## 9. Training Loss Visualization

Plot Generator and Discriminator losses over the course of training to assess convergence.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), d_losses, label='Discriminator Loss', color='red')
plt.plot(range(1, NUM_EPOCHS + 1), g_losses, label='Generator Loss',     color='blue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SAGAN Training Losses')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('training_losses.png', dpi=150)
plt.show()
print('Loss plot saved to training_losses.png')

---
## 10. Results and Visualization

Generate new images using the trained Generator and visualize the self-attention maps.

In [ ]:
def generate_images(generator: nn.Module, num_images: int = 16,
                    z_dim: int = Z_DIM, device: torch.device = device):
    """Generate a grid of images using the trained Generator.

    Args:
        generator:  Trained Generator model.
        num_images: Number of images to generate.
        z_dim:      Latent vector dimensionality.
        device:     Compute device.

    Returns:
        Matplotlib figure with the image grid.
    """
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_images, z_dim, device=device)
        fake_images, attn1, attn2 = generator(z)

    # Denormalize
    fake_images = (fake_images + 1) / 2
    grid = make_grid(fake_images.cpu(), nrow=4, padding=2)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid.permute(1, 2, 0))
    ax.set_title('Generated Images (SAGAN)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    return fig

fig = generate_images(G)

In [ ]:
def visualize_attention(generator: nn.Module, z_dim: int = Z_DIM,
                        device: torch.device = device):
    """Visualize the self-attention maps produced by the Generator.

    Displays the attention weight matrices from both self-attention layers.

    Args:
        generator: Trained Generator model.
        z_dim:     Latent vector dimensionality.
        device:    Compute device.
    """
    generator.eval()
    with torch.no_grad():
        z = torch.randn(1, z_dim, device=device)
        _, attn1, attn2 = generator(z)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, attn, title in zip(axes,
                                [attn1, attn2],
                                ['Self-Attention Map (layer 1)',
                                 'Self-Attention Map (layer 2)']):
        # Show attention from the first spatial position (index 0)
        attn_np = attn[0].cpu().numpy()  # (N, N)
        im = ax.imshow(attn_np, cmap='viridis', aspect='auto')
        ax.set_title(title)
        ax.set_xlabel('Key position')
        ax.set_ylabel('Query position')
        plt.colorbar(im, ax=ax)

    plt.suptitle('Generator Self-Attention Maps', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_attention(G)

---
## 11. Model Evaluation

Compute the **Fréchet Inception Distance (FID)** score to quantitatively evaluate the quality of generated images.

> **Note:** FID computation requires the `pytorch-fid` package (`pip install pytorch-fid`).

In [ ]:
# Generate a full set of evaluation images and save them to disk
EVAL_DIR = 'eval_images'
os.makedirs(EVAL_DIR, exist_ok=True)

NUM_EVAL_IMAGES = 1000  # Number of images for FID computation

G.eval()
with torch.no_grad():
    generated_count = 0
    while generated_count < NUM_EVAL_IMAGES:
        z = torch.randn(min(BATCH_SIZE, NUM_EVAL_IMAGES - generated_count),
                        Z_DIM, device=device)
        imgs, _, _ = G(z)
        imgs = (imgs + 1) / 2  # Denormalize to [0, 1]
        for i, img in enumerate(imgs):
            save_image(img, os.path.join(EVAL_DIR, f'gen_{generated_count + i:05d}.png'))
        generated_count += imgs.size(0)

print(f'Saved {NUM_EVAL_IMAGES} generated images to "{EVAL_DIR}/"')
print('Run: python -m pytorch_fid data/ eval_images/ --device cuda')

---
## Summary

This notebook demonstrated a complete SAGAN training pipeline:

| Component               | Details                                          |
|-------------------------|--------------------------------------------------|
| Architecture            | SAGAN (Self-Attention GAN)                       |
| Attention mechanism     | Spatial self-attention in G and D                |
| Normalization           | Spectral normalization (G & D), BatchNorm (G)    |
| Loss function           | Hinge loss                                       |
| Optimizer               | Adam (β₁=0.0, β₂=0.9)                          |
| Image resolution        | 64 × 64                                          |
| Evaluation metric       | Fréchet Inception Distance (FID)                 |

**References:**
- Zhang, H. et al. (2019). *Self-Attention Generative Adversarial Networks.* ICML 2019. [arXiv:1805.08318](https://arxiv.org/abs/1805.08318)
- Miyato, T. et al. (2018). *Spectral Normalization for Generative Adversarial Networks.* ICLR 2018. [arXiv:1802.05957](https://arxiv.org/abs/1802.05957)